# 🌸 CodeAlpha Data Science Internship - Task 1
## Iris Flower Classification with Machine Learning

**Author:** CodeAlpha Intern  
**Domain:** Data Science & Machine Learning  
**Task Description:**  
The Iris flower classification problem is one of the most famous benchmark datasets in pattern recognition and machine learning. In this project, we train supervised machine learning models to classify iris flowers into three distinct species (**Setosa**, **Versicolor**, and **Virginica**) using morphological measurements:
1. Sepal Length (cm)
2. Sepal Width (cm)
3. Petal Length (cm)
4. Petal Width (cm)

---
### 📌 Workflow Overview:
1. **Data Ingestion & Integrity Check**: Load dataset, inspect null values, types, and summary statistics.
2. **Exploratory Data Analysis (EDA)**: Multivariate pairplots, distributions, and correlation analysis.
3. **Feature Scaling & Data Splitting**: Stratified 80/20 train-test split and standard scaling.
4. **Model Exploration & Benchmarking**:
   - Logistic Regression
   - Support Vector Classifier (RBF Kernel)
   - Decision Tree Classifier
   - Random Forest Classifier
   - K-Nearest Neighbors (KNN)
5. **Stratified 5-Fold Cross-Validation**: Rigorous evaluation to ensure robustness and avoid overfitting.
6. **Detailed Evaluation**: Confusion matrix, classification report, precision/recall/F1, and 2D PCA decision boundary visualization.
7. **Model Deployment / Serialization**: Saving trained model pipeline via `joblib`.


In [ ]:
# Import required libraries
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.decomposition import PCA

# Plotting settings
sns.set_theme(style="whitegrid", palette="muted")
%matplotlib inline


### 1. Data Ingestion & Preprocessing

In [ ]:
# Load the dataset
dataset_path = os.path.join("dataset", "iris.csv")
df = pd.read_csv(dataset_path)

if 'Id' in df.columns:
    df = df.drop(columns=['Id'])

print(f"Dataset Shape: {df.shape}")
df.head()


In [ ]:
# Data Summary & Missing Values Check
print("Missing values per column:\n", df.isnull().sum())
print("\nClass distribution:\n", df['Species'].value_counts())
df.describe()


### 2. Exploratory Data Analysis (EDA)

In [ ]:
# Pairplot to explore distributions and separability
pairplot = sns.pairplot(df, hue='Species', palette='viridis', corner=False, diag_kind='kde')
pairplot.fig.subplots_adjust(top=0.93)
pairplot.fig.suptitle("Iris Morphological Feature Distributions by Species", fontsize=14, fontweight='bold')
plt.show()


In [ ]:
# Correlation Matrix Heatmap
plt.figure(figsize=(8, 5))
numeric_df = df.select_dtypes(include=[np.number])
sns.heatmap(numeric_df.corr(), annot=True, cmap='Blues', fmt=".3f", linewidths=1.0)
plt.title("Correlation Matrix of Morphological Features", fontsize=14, fontweight='bold')
plt.show()


### 3. Feature Scaling & Data Partitioning

In [ ]:
# Partition features and target
X = df.drop(columns=['Species'])
y = df['Species']

# Stratified 80/20 train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Feature Standardization
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set: {X_train.shape[0]} samples | Testing set: {X_test.shape[0]} samples")


### 4. Machine Learning Model Comparison & Cross-Validation

In [ ]:
# Initialize candidate classifiers
models = {
    "Logistic Regression": LogisticRegression(max_iter=500, random_state=42),
    "Support Vector Machine": SVC(kernel='rbf', probability=True, random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=4, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=4, random_state=42),
    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=5)
}

results = []
trained_models = {}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='accuracy')
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    
    results.append({
        "Model": name,
        "CV Accuracy (Mean)": cv_scores.mean(),
        "CV Std": cv_scores.std(),
        "Test Accuracy": acc
    })
    trained_models[name] = model

results_df = pd.DataFrame(results).sort_values(by="Test Accuracy", ascending=False)
results_df


### 5. Detailed Best Model Evaluation

In [ ]:
# Evaluate Best Model
best_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_name]
y_pred_best = best_model.predict(X_test_scaled)

print(f"=== Best Model: {best_name} ===")
print(classification_report(y_test, y_pred_best))

# Confusion Matrix Heatmap
cm = confusion_matrix(y_test, y_pred_best, labels=best_model.classes_)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=best_model.classes_, yticklabels=best_model.classes_)
plt.title(f"Confusion Matrix ({best_name})", fontweight='bold')
plt.xlabel("Predicted Species")
plt.ylabel("Actual Species")
plt.show()


### 6. 2D PCA Decision Boundary Visualization

In [ ]:
# PCA 2D projection
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_train_scaled)
svm_2d = SVC(kernel='rbf', random_state=42)
svm_2d.fit(X_pca, y_train)

h = .02
x_min, x_max = X_pca[:, 0].min() - 1, X_pca[:, 0].max() + 1
y_min, y_max = X_pca[:, 1].min() - 1, X_pca[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

species_list = list(best_model.classes_)
Z = svm_2d.predict(np.c_[xx.ravel(), yy.ravel()])
Z = np.array([species_list.index(s) for s in Z]).reshape(xx.shape)

plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.coolwarm)
for species in species_list:
    idx = (y_train == species)
    plt.scatter(X_pca[idx, 0], X_pca[idx, 1], label=species, edgecolors='k', s=60)
plt.title("Iris Decision Boundaries in 2D PCA Space", fontweight='bold')
plt.xlabel(f"PCA Component 1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
plt.ylabel(f"PCA Component 2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
plt.legend()
plt.show()


### 7. Pipeline Serialization & Inference Test

In [ ]:
# Save pipeline
pipeline = {
    "model": best_model,
    "scaler": scaler,
    "classes": best_model.classes_,
    "feature_names": list(X.columns)
}
joblib.dump(pipeline, os.path.join("outputs", "iris_model.joblib"))

# Test inference function
def predict_flower(sepal_len, sepal_wid, petal_len, petal_wid):
    loaded = joblib.load(os.path.join("outputs", "iris_model.joblib"))
    features = np.array([[sepal_len, sepal_wid, petal_len, petal_wid]])
    features_scaled = loaded["scaler"].transform(features)
    pred = loaded["model"].predict(features_scaled)[0]
    probs = loaded["model"].predict_proba(features_scaled)[0]
    return pred, dict(zip(loaded["classes"], [round(float(p), 4) for p in probs]))

print("Inference Test:")
sample = (5.8, 2.8, 5.1, 2.4)
pred, probs = predict_flower(*sample)
print(f"Sample {sample} => Predicted Species: {pred}\nClass Probabilities: {probs}")
